##  This notebook provides data needed.
If anyone wants to make a comparison easily they can prepare their data in the format of pickles for test here and compare our benchmarks with their works.

In [2]:
"""
=================== Import required libraries ===================
"""

import time
import csv
import ast
import os
import yaml
import pickle
import random
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import joypy

import seaborn as sns
# import more_itertools
# import geopandas as gpd
import matplotlib as mpl
from pathlib import Path
from typing import Tuple
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.cm import get_cmap
from matplotlib.patches import Patch
import matplotlib.ticker as ticker
from matplotlib.image import imread

from matplotlib.colors import ListedColormap
from scipy.stats import wilcoxon
from scipy.stats import mannwhitneyu
from scipy.stats import f_oneway

from matplotlib.ticker import FormatStrFormatter
from scipy.stats import gaussian_kde  

In [3]:
'''
    To load all simulations for two modelling approaches
    (Ensemble of Top 10 configs post Random search (final 10 models trained each on 10 random seeds)
    and Ensemble of Top-3 regional configs in each of the 6 clusters (final 18 models, trained each on 10 random seeds))
    and the Benchmark from Kratzert 2024
'''

def find_dir_up(start: Path, dirname: str) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        d = p / dirname
        if d.exists() and d.is_dir():
            return d
    raise FileNotFoundError(f"Could not find '{dirname}' from {start}")

PICKLE_DIR = find_dir_up(Path.cwd().parent, "Pickles")
print("Using PICKLE_DIR:", PICKLE_DIR)

def load_pickle(p: Path):
    with p.open("rb") as f:
        return pickle.load(f)

ntLSTMsonSBs     = load_pickle(PICKLE_DIR / "ntLSTMsonSBs_CAMELSUS_test.p")
All_Top10        = load_pickle(PICKLE_DIR / "All_Top10_Configs_CAMELSUS_test.p")
All_Cluster_wise = load_pickle(PICKLE_DIR / "All_Cluster_wise_Configs_CAMELSUS_test.p")


Using PICKLE_DIR: F:\Experiments\CAMELS_US\Clean_4_upload\Pickles


In [6]:
# To check Never Train an LSTM on a Single Basin Paper Predictions (Benchmark) simulations shape

print(ntLSTMsonSBs.keys())
print(ntLSTMsonSBs['01022500'].keys())
print(ntLSTMsonSBs['01022500']['191889'].keys())
print(ntLSTMsonSBs['01022500']['191889']['QObs(mm/d)_sim'])



dict_keys(['01022500', '01031500', '01047000', '01052500', '01054200', '01055000', '01057000', '01073000', '01078000', '01123000', '01134500', '01137500', '01139000', '01139800', '01142500', '01144000', '01162500', '01169000', '01170100', '01181000', '01187300', '01195100', '01333000', '01350000', '01350080', '01350140', '01365000', '01411300', '01413500', '01414500', '01415000', '01423000', '01434025', '01435000', '01439500', '01440000', '01440400', '01451800', '01466500', '01484100', '01487000', '01491000', '01510000', '01516500', '01518862', '01532000', '01539000', '01542810', '01543000', '01543500', '01544500', '01545600', '01547700', '01548500', '01549500', '01550000', '01552000', '01552500', '01557500', '01567500', '01568000', '01580000', '01583500', '01586610', '01591400', '01594950', '01596500', '01605500', '01606500', '01632000', '01632900', '01634500', '01638480', '01639500', '01644000', '01664000', '01666500', '01667500', '01669000', '01669520', '02011400', '02013000', '0201

In [7]:
# To check All Top 10 Predictions simulations shape

print(All_Top10.keys())
print(All_Top10['191889'].keys())
print(All_Top10['191889']['01022500'].keys())
print(All_Top10['191889']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['473461']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['328237']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['643969']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['985957']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['993061']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['307443']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['385445']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['845563']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['665980']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Top10['191889']['01022500']['QObs(mm/d)_sim']['0201_001020'])
print(All_Top10['473461']['01022500']['QObs(mm/d)_sim']['0201_001014'])


dict_keys(['328237', '985957', '473461', '993061', '307443', '385445', '845563', '643969', '665980', '191889'])
dict_keys(['01022500', '01031500', '01047000', '01052500', '01054200', '01055000', '01057000', '01073000', '01078000', '01123000', '01134500', '01137500', '01139000', '01139800', '01142500', '01144000', '01162500', '01169000', '01170100', '01181000', '01187300', '01195100', '01333000', '01350000', '01350080', '01350140', '01365000', '01411300', '01413500', '01414500', '01415000', '01423000', '01434025', '01435000', '01439500', '01440000', '01440400', '01451800', '01466500', '01484100', '01487000', '01491000', '01510000', '01516500', '01518862', '01532000', '01539000', '01542810', '01543000', '01543500', '01544500', '01545600', '01547700', '01548500', '01549500', '01550000', '01552000', '01552500', '01557500', '01567500', '01568000', '01580000', '01583500', '01586610', '01591400', '01594950', '01596500', '01605500', '01606500', '01632000', '01632900', '01634500', '01638480', '

In [8]:
# To check All Cluster_wise Predictions simulations shape

print(All_Cluster_wise.keys())
print(All_Cluster_wise['191889'].keys())
print(All_Cluster_wise['191889']['01022500'].keys())
print(All_Cluster_wise['191889']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['473461']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['328237']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['643969']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['985957']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['993061']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['307443']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['385445']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['845563']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['665980']['01022500']['QObs(mm/d)_sim'].keys())
print(All_Cluster_wise['191889']['01022500']['QObs(mm/d)_sim']['0201_001020'])
print(All_Cluster_wise['473461']['01022500']['QObs(mm/d)_sim']['0201_001014'])


dict_keys(['328237', '985957', '473461', '993061', '307443', '385445', '845563', '643969', '665980', '191889'])
dict_keys(['01022500', '01031500', '01047000', '01052500', '01054200', '01055000', '01057000', '01073000', '01078000', '01123000', '01134500', '01137500', '01139000', '01139800', '01142500', '01144000', '01162500', '01169000', '01170100', '01181000', '01187300', '01195100', '01333000', '01350000', '01350080', '01350140', '01365000', '01411300', '01413500', '01414500', '01415000', '01423000', '01434025', '01435000', '01439500', '01440000', '01440400', '01451800', '01466500', '01484100', '01487000', '01491000', '01510000', '01516500', '01518862', '01532000', '01539000', '01542810', '01543000', '01543500', '01544500', '01545600', '01547700', '01548500', '01549500', '01550000', '01552000', '01552500', '01557500', '01567500', '01568000', '01580000', '01583500', '01586610', '01591400', '01594950', '01596500', '01605500', '01606500', '01632000', '01632900', '01634500', '01638480', '

In [ ]:
"""
To make ensemble models out of all predictions by regional models in each ensemble.
We have developed two ensembles:
1. From Top 10 Regional Configs on the whole CAMELS US 531 basins
2. From Selection of Top 3 Regional Configs on each of the 6 cluaters of the CAMELS US 531 basins
Then this code makes ensemble Median of the predictions of all selected configs in each ensemble on every seed
At the end, every ensemble model has 10 predictions on each of the 10 seeds the same as for the benchmark by Kratzert2024
If anyone wants to compare their predictions and works in the future, they can train their models on the same 10 Random Seeds
and generate a set of 10 predictions each on one of the 10 seeds in every basin. Seeds here only help us understand robustness
of the models. Anyone also can compare only one prediction series by the final median of the 10 predictions of our benchmarks on the 10 seeds.
If anyone wants to compare only one timeserie, they should only focus on the ensemble median of the 10 seeds.
It should be generated. Even they can choose only one seed and work with the ensemble predictions of that specific seed. If so,
we suggest to train thier model on the same seed.
"""

def find_dir_up(start: Path, dirname: str) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        d = p / dirname
        if d.exists() and d.is_dir():
            return d
    raise FileNotFoundError(f"Could not find '{dirname}' from: {start}")


def robust_agg(name: str) -> Callable[[np.ndarray, int], np.ndarray]:
    name = name.lower().strip()
    if name == "median":
        return lambda a, axis: np.nanmedian(a, axis=axis)
    if name == "mean":
        return lambda a, axis: np.nanmean(a, axis=axis)
    if name == "trimmed_mean":
        def _trim(a, axis=0, trim=0.1):
            a = np.sort(a, axis=axis)
            n = a.shape[axis]
            k = int(np.floor(trim * n))
            sl = [slice(None)] * a.ndim
            sl[axis] = slice(k, n - k)
            return np.nanmean(a[tuple(sl)], axis=axis)
        return _trim
    raise ValueError(f"Unknown agg='{name}'. Use 'median', 'mean', or 'trimmed_mean'.")


def build_ensemble_basins_first(
    predictions: Dict[str, Dict[str, Dict[str, Any]]],
    *,
    variable_hint: Optional[str] = None,
    agg: str = "median",
    out_name: str = "Ensemble(mm/d)_sim",
    keep: str = "all",
    verbose: bool = True,
) -> Tuple[Dict[str, Dict[str, Dict[str, xr.DataArray]]], Dict[str, Any]]:
    agg_fn = robust_agg(agg)

    ensemble_result: Dict[str, Dict[str, Dict[str, xr.DataArray]]] = {}
    report = {
        "seeds_total": 0,
        "basins_total": 0,
        "basin_seed_built": 0,
        "skipped_no_data": 0,
        "skipped_shape_mismatch": 0,
        "shape_mismatch_examples": [],
        "other_errors": 0,
        "other_error_examples": [],
    }

    seeds = list(predictions.keys())
    report["seeds_total"] = len(seeds)

    for seed in seeds:
        basins = list(predictions[seed].keys())
        report["basins_total"] += len(basins)

        for basin in basins:
            sims = []
            ref_da: Optional[xr.DataArray] = None
            ref_shape: Optional[Tuple[int, ...]] = None

            try:
                for subfolder, sim_data in predictions[seed][basin].items():
                    if sim_data is None:
                        continue

                    items = sim_data.items() if isinstance(sim_data, dict) else [("data", sim_data)]

                    for k, v in items:
                        if variable_hint is not None and k != variable_hint:
                            continue
                        if not isinstance(v, xr.DataArray):
                            continue

                        if ref_da is None:
                            ref_da = v
                            ref_shape = v.values.shape

                        arr = v.values
                        if ref_shape is not None and arr.shape != ref_shape:
                            if keep == "strict":
                                raise ValueError(f"shape mismatch: {arr.shape} != {ref_shape}")
                            report["skipped_shape_mismatch"] += 1
                            if len(report["shape_mismatch_examples"]) < 10:
                                report["shape_mismatch_examples"].append(
                                    {"seed": seed, "basin": basin, "subfolder": subfolder,
                                     "key": k, "shape": arr.shape, "ref_shape": ref_shape}
                                )
                            continue

                        sims.append(arr)

                if not sims or ref_da is None:
                    report["skipped_no_data"] += 1
                    continue

                stack = np.stack(sims, axis=0)
                ens = agg_fn(stack, 0)

                ensemble_da = xr.DataArray(
                    ens,
                    dims=ref_da.dims,
                    coords=ref_da.coords,
                    name=out_name,
                    attrs=dict(ref_da.attrs) if ref_da.attrs else {},
                )
                ensemble_da.attrs.update({"ensemble_agg": agg, "n_members": int(stack.shape[0])})

                # Basins -> Seeds -> Preds
                ensemble_result.setdefault(basin, {})
                if seed in ensemble_result[basin]:
                    raise KeyError(f"Duplicate basin/seed encountered: basin={basin!r}, seed={seed!r}")
                ensemble_result[basin][seed] = {"ensemble": ensemble_da}
                report["basin_seed_built"] += 1

            except Exception as e:
                report["other_errors"] += 1
                if len(report["other_error_examples"]) < 10:
                    report["other_error_examples"].append({"seed": seed, "basin": basin, "error": repr(e)})

    if verbose:
        print(
            f"Ensemble built basin×seed: {report['basin_seed_built']} | "
            f"seeds={report['seeds_total']} | basins_seen={report['basins_total']} | "
            f"skipped_no_data={report['skipped_no_data']} | "
            f"shape_member_skips={report['skipped_shape_mismatch']} | "
            f"other_errors={report['other_errors']}"
        )

    return ensemble_result, report


def save_pickle(obj: Any, out_path: Path) -> None:
    out_path = out_path.resolve()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


# ---------------------------
# Correct usage: UNPACK and save only dict
# ---------------------------

PICKLE_DIR = find_dir_up(Path.cwd().parent, "Pickles")
print("Using PICKLE_DIR:", PICKLE_DIR)

# Top-10 ensemble
ens_top10_dict, rep_top10 = build_ensemble_basins_first(
    All_Top10,
    variable_hint=None,  # or "QObs(mm/d)_sim"
    agg="median",
    out_name="Ensemble(mm/d)_sim",
    keep="all",
    verbose=True,
)
save_pickle(ens_top10_dict, PICKLE_DIR / "ensemble_Top10_CAMELSUS_test.p")
# optional:
save_pickle(rep_top10,     PICKLE_DIR / "report_ensemble_Top10_CAMELSUS_test.p")

# Cluster-wise ensemble
ens_cluster_dict, rep_cluster = build_ensemble_basins_first(
    All_Cluster_wise,
    variable_hint=None,
    agg="median",
    out_name="Ensemble(mm/d)_sim",
    keep="all",
    verbose=True,
)
save_pickle(ens_cluster_dict, PICKLE_DIR / "ensemble_Cluster_wise_CAMELSUS_test.p")
# optional:
save_pickle(rep_cluster,      PICKLE_DIR / "report_ensemble_Cluster_wise_CAMELSUS_test.p")

print("✅ Saved ensemble dicts (Basins>Seeds>Preds) in:", PICKLE_DIR)


Using PICKLE_DIR: F:\Experiments\CAMELS_US\Clean_4_upload\Pickles
Ensemble built basin×seed: 5310 | seeds=10 | basins_seen=5310 | skipped_no_data=0 | shape_member_skips=0 | other_errors=0
Ensemble built basin×seed: 5310 | seeds=10 | basins_seen=5310 | skipped_no_data=0 | shape_member_skips=0 | other_errors=0
✅ Saved ensemble dicts (Basins>Seeds>Preds) in: F:\Experiments\CAMELS_US\Clean_4_upload\Pickles


In [ ]:
'''
    To load all Ensemble simulations for two modelling approaches and Obs data
'''

Top10     = load_pickle(PICKLE_DIR / "ensemble_Top10_CAMELSUS_test.p")
Cluster_wise        = load_pickle(PICKLE_DIR / "ensemble_Cluster_wise_CAMELSUS_test.p")
Obs_loaded = load_pickle(PICKLE_DIR / "Obs_test.p")


In [16]:
# To check Top10 Ensemble Predictions simulations shape

print(Top10.keys())
print(Top10['01022500'].keys())
print(Top10['01022500']['191889'].keys())
print(Top10['01022500']['191889']['ensemble'])



dict_keys(['01022500', '01031500', '01047000', '01052500', '01054200', '01055000', '01057000', '01073000', '01078000', '01123000', '01134500', '01137500', '01139000', '01139800', '01142500', '01144000', '01162500', '01169000', '01170100', '01181000', '01187300', '01195100', '01333000', '01350000', '01350080', '01350140', '01365000', '01411300', '01413500', '01414500', '01415000', '01423000', '01434025', '01435000', '01439500', '01440000', '01440400', '01451800', '01466500', '01484100', '01487000', '01491000', '01510000', '01516500', '01518862', '01532000', '01539000', '01542810', '01543000', '01543500', '01544500', '01545600', '01547700', '01548500', '01549500', '01550000', '01552000', '01552500', '01557500', '01567500', '01568000', '01580000', '01583500', '01586610', '01591400', '01594950', '01596500', '01605500', '01606500', '01632000', '01632900', '01634500', '01638480', '01639500', '01644000', '01664000', '01666500', '01667500', '01669000', '01669520', '02011400', '02013000', '0201

In [17]:
# To check Cluster_wise Ensemble Predictions simulations shape

print(Cluster_wise.keys())
print(Cluster_wise['01022500'].keys())
print(Cluster_wise['01022500']['191889'].keys())
print(Cluster_wise['01022500']['191889']['ensemble'])



dict_keys(['01022500', '01031500', '01047000', '01052500', '01054200', '01055000', '01057000', '01073000', '01078000', '01123000', '01134500', '01137500', '01139000', '01139800', '01142500', '01144000', '01162500', '01169000', '01170100', '01181000', '01187300', '01195100', '01333000', '01350000', '01350080', '01350140', '01365000', '01411300', '01413500', '01414500', '01415000', '01423000', '01434025', '01435000', '01439500', '01440000', '01440400', '01451800', '01466500', '01484100', '01487000', '01491000', '01510000', '01516500', '01518862', '01532000', '01539000', '01542810', '01543000', '01543500', '01544500', '01545600', '01547700', '01548500', '01549500', '01550000', '01552000', '01552500', '01557500', '01567500', '01568000', '01580000', '01583500', '01586610', '01591400', '01594950', '01596500', '01605500', '01606500', '01632000', '01632900', '01634500', '01638480', '01639500', '01644000', '01664000', '01666500', '01667500', '01669000', '01669520', '02011400', '02013000', '0201

In [18]:
# To check Observations shape

print(Obs_loaded.keys())
print(Obs_loaded['01022500'])



dict_keys(['01022500', '01031500', '01047000', '01052500', '01054200', '01055000', '01057000', '01073000', '01078000', '01123000', '01134500', '01137500', '01139000', '01139800', '01142500', '01144000', '01162500', '01169000', '01170100', '01181000', '01187300', '01195100', '01333000', '01350000', '01350080', '01350140', '01365000', '01411300', '01413500', '01414500', '01415000', '01423000', '01434025', '01435000', '01439500', '01440000', '01440400', '01451800', '01466500', '01484100', '01487000', '01491000', '01510000', '01516500', '01518862', '01532000', '01539000', '01542810', '01543000', '01543500', '01544500', '01545600', '01547700', '01548500', '01549500', '01550000', '01552000', '01552500', '01557500', '01567500', '01568000', '01580000', '01583500', '01586610', '01591400', '01594950', '01596500', '01605500', '01606500', '01632000', '01632900', '01634500', '01638480', '01639500', '01644000', '01664000', '01666500', '01667500', '01669000', '01669520', '02011400', '02013000', '0201